# 🥈 Silver Layer - Data Transformation | Transformação dos Dados

---

# 🇧🇷 Português

## 📌 Objetivo

A camada **Silver** é responsável pela limpeza, padronização e enriquecimento dos dados provenientes da camada Bronze.

Nesta etapa são aplicadas regras de negócio, validações e transformações que tornam os dados consistentes e preparados para análises.

Diferentemente da Bronze, onde os dados permanecem em seu estado original, a Silver entrega um conjunto de dados confiável para utilização na camada Gold.

# 🇺🇸 English

## 📌 Purpose

The **Silver Layer** is responsible for cleaning, standardizing and enriching the data received from the Bronze Layer.

Business rules, validations and feature engineering are applied to transform raw data into reliable datasets ready for analytical processing.

Unlike the Bronze Layer, where data remains unchanged, the Silver Layer provides trusted and standardized information for the Gold Layer.

---

## 🛠 Tecnologias Utilizadas/Technologies

- Databricks
- Apache Spark (PySpark)
- SQL
- Delta Lake

---

## 📂 Dados de Entrada/Input Tables

### Bronze Layer

- bronze.products
- bronze.categories

---

## ⚙️ Transformações Aplicadas/⚙️ Transformations

✔ Product and Category JOIN

✔ Duplicate removal

✔ Category enrichment

✔ Discount amount calculation

✔ Discount percentage calculation

✔ Price range classification

✔ Product rating classification

✔ Weighted Rating calculation

✔ Best Seller indicators

✔ ETL execution timestamp

---

Nesta etapa foram realizadas as seguintes transformações:

✔ JOIN entre produtos e categorias

✔ Remoção de registros duplicados

✔ Enriquecimento da base com o nome das categorias

✔ Cálculo do valor do desconto

✔ Cálculo do percentual de desconto

✔ Classificação da faixa de preço

✔ Classificação das avaliações

✔ Cálculo do Rating Ponderado

✔ Criação de indicadores para Best Sellers

✔ Inclusão do timestamp de execução do ETL

---

## 📦 Dados de Saída/Output

Schema:

silver

Tabela:

silver.products

---

## 📐 Fluxo/Structure

Bronze Layer
        │
        ▼
Data Cleaning
        │
        ▼
Business Rules
        │
        ▼
Feature Engineering
        │
        ▼
Silver Layer

---

## ✅ Resultado Esperado

Ao final desta etapa, os dados encontram-se limpos, enriquecidos e estruturados para consumo analítico na camada Gold.

## ✅ Expected Result

The Silver Layer delivers clean, standardized and enriched datasets ready for business analytics and Gold Layer modeling.

## 🔧 Correções v2 | Fixes v2

**🇧🇷** Esta versão corrige quatro problemas identificados na análise dos dados:

1. `discount_amount` e `discount_percentage` agora só são calculados quando `price > 0` **e** `listPrice > price`; caso contrário valem 0. Isso elimina descontos negativos (produtos sem preço de lista) e "descontos" de 100% em itens com `price = 0`.
2. Nova classe `Unrated` em `rating_category` para produtos com `stars = 0` (antes caíam em `Poor`).
3. Células de validação ao final da camada Silver.
4. As views do dashboard (notebooks 04 e 05) agora são **persistentes** (`gold.vw_dashboard_*`), em vez de `TEMP VIEW`.

**🇺🇸** This version fixes four issues found during data analysis: discounts are computed only when `price > 0` and `listPrice > price` (no negative or 100% fake discounts); a new `Unrated` class for products with `stars = 0`; validation checks at the end of Silver; and dashboard views are now persistent (`gold.vw_dashboard_*`) instead of `TEMP VIEW`.

In [ ]:
from pyspark.sql.functions import (
    col,
    when,
    round,
    log10,
    current_timestamp
)

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [ ]:
products = spark.table("bronze.products")
categories = spark.table("bronze.categories")

In [ ]:
products.printSchema()
categories.printSchema()

In [ ]:
display(products.limit(10))
display(categories.limit(10))

In [ ]:
silver_df = (
    products.alias("p")
    .join(
        categories.alias("c"),
        col("p.category_id") == col("c.id"),
        "left"
    )
)

In [ ]:
silver_df = silver_df.dropDuplicates(["asin"])

In [ ]:
silver_df = silver_df.withColumn(
    "discount_amount",
    when(
        (col("price") > 0) & (col("listPrice") > col("price")),
        round(col("listPrice") - col("price"), 2)
    ).otherwise(0.0)
)

In [ ]:
silver_df = silver_df.withColumn(
    "discount_percentage",
    when(
        (col("price") > 0) & (col("listPrice") > col("price")),
        round(
            (
                (col("listPrice") - col("price"))
                / col("listPrice")
            ) * 100,
            2
        )
    ).otherwise(0.0)
)

In [ ]:
silver_df = silver_df.withColumn(
    "is_discounted",
    when(
        col("discount_amount") > 0,
        "Yes"
    ).otherwise("No")
)

In [ ]:
silver_df = silver_df.withColumn(
    "price_range",
    when(col("price") < 50, "Low")
    .when(col("price") < 200, "Medium")
    .otherwise("High")
)

In [ ]:
silver_df = silver_df.withColumn(
    "rating_category",
    when(col("stars") == 0, "Unrated")
    .when(col("stars") >= 4.5, "Excellent")
    .when(col("stars") >= 4.0, "Good")
    .when(col("stars") >= 3.0, "Average")
    .otherwise("Poor")
)

In [ ]:
silver_df = silver_df.withColumn(
    "weighted_rating",
    round(
        col("stars") * log10(col("reviews") + 1),
        2
    )
)

In [ ]:
silver_df = silver_df.withColumn(
    "best_seller_flag",
    when(col("isBestSeller") == True, "Yes")
    .otherwise("No")
)

In [ ]:
silver_df = silver_df.withColumn(
    "etl_timestamp",
    current_timestamp()
)

In [ ]:
silver_df = silver_df.select(

    "asin",

    "title",

    "category_name",

    "stars",

    "reviews",

    "price",

    "listPrice",

    "discount_amount",

    "discount_percentage",

    "price_range",

    "rating_category",

    "weighted_rating",

    "best_seller_flag",

    "boughtInLastMonth",

    "is_discounted",

    "productURL",

    "imgUrl",

    "etl_timestamp"

)

In [ ]:
display(silver_df)

In [ ]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.products")
)

In [ ]:
%sql
SELECT *
FROM silver.products
LIMIT 20;

In [ ]:
print(f"Produtos Silver: {silver_df.count()}")

In [ ]:
%sql
-- Validação v2: nenhuma anomalia de desconto deve restar
SELECT
    COUNT(*)                                                              AS total_products,
    COUNT(DISTINCT asin)                                                  AS unique_asin,
    SUM(CASE WHEN discount_amount < 0 THEN 1 ELSE 0 END)                  AS negative_discount_amount,
    SUM(CASE WHEN discount_percentage < 0
              OR discount_percentage >= 100 THEN 1 ELSE 0 END)            AS invalid_discount_pct,
    SUM(CASE WHEN rating_category = 'Unrated' THEN 1 ELSE 0 END)          AS unrated_products,
    SUM(CASE WHEN price = 0 THEN 1 ELSE 0 END)                            AS products_without_price
FROM silver.products;

In [ ]:
display(
    silver_df.describe(
        "price",
        "discount_percentage",
        "stars",
        "weighted_rating"
    )
)

In [ ]:
%sql
SELECT COUNT(*) AS total_produtos
FROM silver.products;

In [ ]:
%sql
SELECT COUNT(DISTINCT category_name) AS total_categorias
FROM silver.products;

In [ ]:
%sql
SELECT
    ROUND(AVG(discount_percentage),2) AS desconto_medio,
    MAX(discount_percentage) AS maior_desconto,
    ROUND(AVG(weighted_rating),2) AS rating_medio
FROM silver.products;

In [ ]:
%sql
SELECT
    price_range,
    COUNT(*) AS quantidade
FROM silver.products
GROUP BY price_range
ORDER BY quantidade DESC;

In [ ]:
%sql
SELECT
    rating_category,
    COUNT(*) AS quantidade
FROM silver.products
GROUP BY rating_category
ORDER BY quantidade DESC;